# Fire an attention graph
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dataflow-arch-for-ai/dataflow-arch-for-ai/blob/main/demos/week2/Dataflow_Firing_Demo_Colab.ipynb)

**Dataflow Architectures for AI, Week 2, Module 1: The dataflow execution model**  
**Author:** Dr. Kaoutar El Maghraoui

The lecture gives three rules for dataflow execution. A computation is a directed acyclic graph of operations. An operation becomes **eligible** when all of its inputs have arrived. The graph decides readiness, and the execution system decides placement, resources and timing.

In this notebook you capture an attention block as an FX graph and execute it under those rules. You find which nodes become eligible together, measure the critical path, track how long each tensor stays alive, and then schedule the same graph on a machine with one matrix engine and on a machine with three. Everything runs on the CPU.

**What you will do**
1. Capture a single-head attention block with `torch.fx`.
2. Group the nodes into firing waves: all nodes eligible at the same step.
3. Find the critical path and compare it with the number of operations.
4. Track each tensor's lifetime and the live memory at each step.
5. Schedule the graph with one and with three matrix engines.

**Before you start:** any runtime works, including CPU. The notebook takes under a minute.

## 1. Capture the attention block
The block follows the lecture: three projections of the same input `x` into `q`, `k` and `v`, a score matmul `q @ k^T`, a scale, a softmax, a second matmul with `v`, and an output projection. It has one head so the graph stays small enough to read.

`torch.fx.symbolic_trace` runs `forward` with placeholder tensors and records each operation as a node.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.fx
from torch.fx.passes.shape_prop import ShapeProp


class Attention(nn.Module):
    def __init__(self, d=256):
        super().__init__()
        self.q_proj = nn.Linear(d, d)
        self.k_proj = nn.Linear(d, d)
        self.v_proj = nn.Linear(d, d)
        self.o_proj = nn.Linear(d, d)
        self.scale = 1.0 / math.sqrt(d)

    def forward(self, x):
        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        probs = torch.softmax(scores, dim=-1)
        out = torch.matmul(probs, v)
        return self.o_proj(out)


SEQ, D = 512, 256
block = Attention(D).eval()
gm = torch.fx.symbolic_trace(block)
x = torch.randn(1, SEQ, D)
ShapeProp(gm).propagate(x)   # attach the shape and dtype of every tensor to its node

for node in gm.graph.nodes:
    shape = tuple(node.meta["tensor_meta"].shape) if "tensor_meta" in node.meta else ""
    inputs = [a.name for a in node.all_input_nodes]
    print(f"{node.op:14} {node.name:10} inputs={inputs!s:24} shape={shape}")

**Reading the output**
- Each line is one node: its type, its name, the nodes it reads, and the shape of the tensor it produces.
- There is no edge list. A node's dependencies are its inputs, as the lecture says: trace the argument references to see the dataflow.
- `q_proj`, `k_proj` and `v_proj` all read only `x`. The Python code wrote them one after another, but the graph has no edge between them.

## 2. Firing waves
Under rule two, a node is eligible as soon as all its inputs exist. If every eligible node ran immediately, each node would run at step `1 + max(step of its inputs)`. Nodes with the same step form a **wave**.

In [ ]:
from collections import defaultdict

ops = [n for n in gm.graph.nodes if n.op not in ("placeholder", "output")]
step = {}
for n in gm.graph.nodes:              # FX lists nodes in a valid execution order
    if n.op == "placeholder":
        step[n] = 0
    elif n.op != "output":
        step[n] = 1 + max((step[a] for a in n.all_input_nodes), default=0)

waves = defaultdict(list)
for n in ops:
    waves[step[n]].append(n.name)
for s in sorted(waves):
    print(f"wave {s}: {', '.join(waves[s])}")

critical_path = max(step.values())
print(f"\n{len(ops)} operations, critical path {critical_path} steps, widest wave {max(len(w) for w in waves.values())} nodes")

**Reading the output**
- Wave 1 holds `q_proj`, `k_proj` and `v_proj`. All three became eligible at the same moment, with no threads or parallel annotations in the code. The concurrency is in the graph.
- The score matmul waits for both `q` and the transposed `k`. It is a **join**: if one branch is slower, the join waits.
- The critical path is the longest chain of dependent nodes. No schedule can finish in fewer steps, however many engines the machine has.

## 3. Lifetimes and live memory
A tensor is alive from the step that produces it to the last step that reads it. On a GPU with large caches and HBM, a long lifetime costs little. On an accelerator with small, explicitly managed scratchpads, every live tensor uses capacity.

The cell computes each tensor's lifetime and the total live bytes at each step, with FP32 values.

In [ ]:
users = defaultdict(list)
for n in ops:
    for a in n.all_input_nodes:
        users[a].append(n)

def nbytes(n):
    m = n.meta["tensor_meta"]
    return math.prod(m.shape) * torch.empty((), dtype=m.dtype).element_size()

tensors = [n for n in gm.graph.nodes if n.op != "output"]
life = {}
for n in tensors:
    last = max((step[u] for u in users[n]), default=step[n])
    life[n] = (step[n], last)

print(f"{'tensor':10} {'born':>4} {'last read':>9} {'span':>4} {'size':>9}")
for n in tensors:
    born, last = life[n]
    print(f"{n.name:10} {born:4d} {last:9d} {last - born:4d} {nbytes(n) / 1024:7.0f} KB")

print("\nLive memory at each step")
for s in range(1, critical_path + 1):
    live = [n for n in tensors if life[n][0] <= s <= life[n][1]]
    total = sum(nbytes(n) for n in live)
    print(f"  step {s}: {total / 1024:6.0f} KB  ({', '.join(n.name for n in live)})")

**Reading the output**
- `v_proj` is produced in wave 1 and read in the second matmul, several steps later. It has the longest span of any intermediate tensor, as the lecture points out.
- `v` is not on the critical path. Computing it earlier does not finish the block sooner, but it holds memory for longer.
- Live memory peaks when the `SEQ` by `SEQ` score and probability tensors exist. At 512 tokens each is 1 MB in FP32. The score tensor grows with the square of the sequence length.

## 4. Eligible is not executed
The graph exposes three eligible projections. Whether they run at the same time depends on the machine. The cell schedules the graph on a simple machine model:

- Matrix work (`Linear` layers and matmuls) runs on matrix engines, one node per engine per step.
- Everything else (transpose, scale, softmax) runs on one vector unit.
- At each step the scheduler starts as many eligible nodes as it has free units.

In [ ]:
def schedule(matrix_engines, vector_units=1):
    def unit(n):
        if n.op == "call_module" or (n.op == "call_function" and n.target is torch.matmul):
            return "matrix"
        return "vector"

    placeholder = next(n for n in gm.graph.nodes if n.op == "placeholder")
    done, t, log = {placeholder}, 0, []
    remaining = list(ops)
    while remaining:
        t += 1
        eligible = [n for n in remaining if all(a in done for a in n.all_input_nodes)]
        free = {"matrix": matrix_engines, "vector": vector_units}
        started = []
        for n in eligible:
            if free[unit(n)] > 0:
                free[unit(n)] -= 1
                started.append(n)
        done.update(started)
        remaining = [n for n in remaining if n not in started]
        log.append((t, [n.name for n in eligible], [n.name for n in started]))
    return t, log


for engines in (1, 3):
    steps, log = schedule(engines)
    print(f"--- {engines} matrix engine(s): {steps} steps")
    for t, eligible, started in log:
        waiting = [e for e in eligible if e not in started]
        print(f"  step {t}: run {', '.join(started):30}" + (f" waiting: {', '.join(waiting)}" if waiting else ""))

**Reading the output**
- With one matrix engine, the three projections are eligible together but run one after another. The block takes one extra step, not two: in step 3 the vector unit runs `transpose` while the matrix engine runs `v_proj`. Overlapping work on different units hides part of the delay.
- With three engines, the projections run in the same step and the schedule matches the critical path.
- The dependencies are the same in both runs. Only the resources changed. This is rule three: the graph gives readiness, the machine gives placement and timing.

## 5. Adjacent nodes, different workloads
The lecture points out that softmax, between two matmuls, has a very different arithmetic intensity. The cell estimates FLOPs and bytes for each node (FP32, inputs and weights read once, output written once) and prints the ratio.

In [ ]:
def flops(n):
    out = n.meta["tensor_meta"].shape
    if n.op == "call_module":                      # Linear: 2 * rows * in * out
        layer = gm.get_submodule(n.target)
        return 2 * math.prod(out[:-1]) * layer.in_features * layer.out_features
    if n.op == "call_function" and n.target is torch.matmul:
        a = n.all_input_nodes[0].meta["tensor_meta"].shape
        return 2 * math.prod(out) * a[-1]
    return 5 * math.prod(out)                      # rough count for elementwise and softmax

def bytes_moved(n):
    total = sum(nbytes(a) for a in n.all_input_nodes) + nbytes(n)
    if n.op == "call_module":                      # a Linear layer also reads its weight and bias
        total += sum(p.numel() * p.element_size() for p in gm.get_submodule(n.target).parameters())
    return total

for n in ops:
    print(f"{n.name:10} {flops(n) / 1e6:9.1f} MFLOP {bytes_moved(n) / 1e6:7.2f} MB  intensity {flops(n) / bytes_moved(n):6.1f} FLOPs/byte")

**Reading the output**
- The projections and matmuls do tens of FLOPs per byte. The scale and softmax do about one.
- On Spyre, the lecture shows the scores staying in the same core while the work moves from the matrix engine to the vector unit. Locality chooses the core, and the operation type chooses the unit.

## Try this
1. Change `SEQ` to 2048 and run sections 1 and 3 again. The score tensor grows 16 times, and so does the peak live memory.
2. In section 4, schedule with two vector units. Does the block finish sooner? Use the waves from section 2 to explain why.
3. Add a residual connection, `return self.o_proj(out) + x`. Which tensor now has the longest lifetime?

## Summary
- A dataflow graph has no program counter. A node is eligible when its inputs exist, so independent nodes become eligible together.
- The critical path is a lower bound on the number of steps. Resources decide whether eligible nodes run together.
- Lifetimes come from the graph. A tensor produced early and read late holds memory across many steps, which matters on scratchpad machines.
- Neighboring nodes can have very different arithmetic intensity and need different engines.

Lecture: Week 2, Module 1, slides 6 to 15 (three rules, attention recap, graph firing steps 1 to 7).